# 📄 Local RAG over `.docx` files — Google Colab free tier (T4)

**100% local:** `BAAI/bge-m3` embeddings → FAISS (cosine) → optional `BAAI/bge-reranker-v2-m3` → `Qwen2.5-7B-Instruct` in 4-bit NF4 → Gradio UI.
No hosted APIs. Greedy decoding (temperature 0 ⇒ `do_sample=False`), answers only from retrieved context, with `[n]` citations.

## How to run
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.**
3. Cell 1 installs pinned libraries (~1–2 min). If Colab shows a **"Restart session"** prompt, accept it, then **Run all** again.
4. Approve **Google Drive** access when asked (used to save/reuse the index and to read `.docx` from `MyDrive/rag_docs/`). Deny it and everything still works, just without persistence.
5. First run downloads ~17 GB of models (5–10 min). Later sessions reuse the HF cache only within the same runtime.
6. When the **upload widget** appears (Cell 13), pick your `.docx` files. Click *Cancel* to skip — the notebook then uses your saved Drive index, or a small built-in demo document.
7. Check the **self-test** output (Cell 15), then open the `https://….gradio.live` link printed by the last cell.

## How to tune it
| What | Where | Guidance |
|---|---|---|
| Chunk size / overlap | `CONFIG["CHUNK_TOKENS"]`, `CONFIG["CHUNK_OVERLAP"]` (Cell 2) | 300–400 for FAQs, policies, precise facts; 700–900 for narrative text. Keep overlap ≈ 15%. Changing it requires re-indexing: **Reset index** in the UI, then **Build index** again. |
| top-k | UI slider, or `CONFIG["TOP_K"]` | 3–5 for precise questions, 8–10 for "summarize / list all" questions. More k = longer prompt and slower answers; the context budget truncates automatically. |
| Reranker | UI checkbox, or `CONFIG["USE_RERANKER"] = True` | Fetches `RERANK_CANDIDATES` (20) by cosine, rescores with a cross-encoder, keeps top-k. Helps a lot when many sections look alike. Costs ~1.1 GB VRAM and ~0.5–1 s per question. Loaded on first use. |
| LLM | `CONFIG["LLM_MODEL"]` / `LLM_FALLBACK`, or Cell 17 `switch_llm(...)` | 7B is the default; it auto-falls back to 3B if VRAM is short. Other T4-friendly 4-bit options: `Qwen/Qwen2.5-3B-Instruct` (faster), `Qwen/Qwen2.5-14B-Instruct` does **not** fit comfortably. Changing the LLM does not require re-indexing. |
| Embedder | `CONFIG["EMBED_MODEL"]` | Changing it **requires re-indexing** (a saved index built with another embedder is refused on load). `intfloat/multilingual-e5-base` is smaller/faster but weaker on Uzbek. |
| "Not found" strictness | `CONFIG["MIN_SCORE"]` | If the best cosine score is below this, the LLM is skipped and the not-found message is returned. bge-m3: unrelated ≈ 0.3–0.45, relevant ≈ 0.55+. Raise to ~0.40 if it answers off-topic questions; lower it if it refuses too often. |
| Prompt size | `CONFIG["CONTEXT_WINDOW_CAP"]` | Qwen2.5 supports 32k, but 8k keeps the KV cache safe on a T4. |
| Exact figures paraphrased? | `CONFIG["REPETITION_PENALTY"]` | 1.1 also penalises tokens that appear in the *context*, which can make the model avoid copying exact numbers/names. Drop to 1.0–1.05 if you see that. |

### Cell 1 — Install pinned libraries (torch is Colab's preinstalled CUDA build)

In [ ]:
# transformers is pinned to the last 4.x line: v5 changed the bitsandbytes 4-bit loading path
# and some kwargs. torch is NOT reinstalled — Colab's build matches its CUDA driver.
!pip install -q --no-warn-conflicts \
    "transformers==4.57.3" \
    "huggingface_hub==0.36.0" \
    "accelerate==1.10.1" \
    "bitsandbytes==0.49.2" \
    "sentence-transformers==5.1.2" \
    "faiss-cpu==1.12.0" \
    "python-docx==1.2.0" \
    "gradio==5.49.1"

import importlib.metadata as _md
print("Installed versions:")
for _p in ["torch", "transformers", "huggingface_hub", "accelerate", "bitsandbytes",
           "sentence-transformers", "faiss-cpu", "python-docx", "gradio"]:
    try:
        print(f"  {_p:22s} {_md.version(_p)}")
    except _md.PackageNotFoundError:
        print(f"  {_p:22s} NOT INSTALLED")

### Cell 2 — Configuration (the only place you need to edit to tune things)

In [ ]:
CONFIG = {
    # ── Models (all downloaded from Hugging Face, run locally) ──
    "EMBED_MODEL":    "BAAI/bge-m3",
    "EMBED_FALLBACK": "intfloat/multilingual-e5-base",
    "RERANK_MODEL":   "BAAI/bge-reranker-v2-m3",
    "LLM_MODEL":      "Qwen/Qwen2.5-7B-Instruct",
    "LLM_FALLBACK":   "Qwen/Qwen2.5-3B-Instruct",
    "MIN_FREE_VRAM_FOR_MAIN_LLM_GB": 7.0,   # below this, skip straight to the fallback LLM

    # ── Chunking (measured in LLM tokens) ──
    "CHUNK_TOKENS":  500,
    "CHUNK_OVERLAP": 80,

    # ── Retrieval ──
    "TOP_K":             5,
    "USE_RERANKER":      False,   # default state of the UI checkbox
    "RERANK_CANDIDATES": 20,      # cosine candidates passed to the reranker
    "MIN_SCORE":         0.20,    # best cosine below this → "not found" without calling the LLM

    # ── Generation (temperature 0 ⇒ greedy, do_sample=False) ──
    "MAX_NEW_TOKENS":     512,
    "REPETITION_PENALTY": 1.1,
    "CONTEXT_WINDOW_CAP": 8192,   # prompt + answer never exceed min(this, model max)

    # ── Storage ──
    "USE_DRIVE":             True,
    "DRIVE_INDEX_DIR":       "/content/drive/MyDrive/rag_index",
    "DRIVE_DOCS_DIR":        "/content/drive/MyDrive/rag_docs",   # optional: .docx here are indexed too
    "AUTO_LOAD_SAVED_INDEX": True,
    "LOCAL_DOCS_DIR":        "/content/rag_docs",
}

NOT_FOUND = "I couldn't find this in the documents."

### Cell 3 — GPU check, imports and memory helpers

In [ ]:
import os, gc, re, json, time, glob, hashlib, zipfile, warnings
import numpy as np
import torch

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not torch.cuda.is_available():
    print("❌ No GPU detected.\n"
          "   Fix: Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save,\n"
          "   then Runtime → Run all.")
    raise RuntimeError("GPU required — see the message above.")

DEVICE = "cuda"
torch.manual_seed(0)

def free_memory():
    """Release Python garbage and cached CUDA blocks."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

def vram_report(tag=""):
    free_b, total_b = torch.cuda.mem_get_info()
    alloc = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    msg = (f"[VRAM {tag}] allocated {alloc:.2f} GB · reserved {reserved:.2f} GB · "
           f"free {free_b/1024**3:.2f} / {total_b/1024**3:.2f} GB")
    try:
        import psutil
        vm = psutil.virtual_memory()
        msg += f" | RAM used {vm.used/1024**3:.1f} / {vm.total/1024**3:.1f} GB"
    except Exception:
        pass
    print(msg)

print(f"✅ GPU: {torch.cuda.get_device_name(0)} · torch {torch.__version__} · CUDA {torch.version.cuda}")
vram_report("start")

### Cell 4 — Mount Google Drive (optional: index persistence + docs folder)

In [ ]:
DRIVE_OK = False
if CONFIG["USE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(CONFIG["DRIVE_INDEX_DIR"], exist_ok=True)
        DRIVE_OK = True
        print(f"✅ Drive mounted. Index folder: {CONFIG['DRIVE_INDEX_DIR']}")
    except Exception as e:
        print(f"⚠️ Drive not mounted ({type(e).__name__}: {e}). Continuing without save/load.")

### Cell 5 — `.docx` loader: paragraphs + tables in document order, heading tracking, error handling

In [ ]:
from docx import Document
from docx.oxml.ns import qn
from docx.table import Table
from docx.text.paragraph import Paragraph
from docx.opc.exceptions import PackageNotFoundError

_HEADING_PREFIXES = ("heading", "title", "заголовок", "sarlavha")

def _heading_level(p):
    """1..9 for headings (style name or outline level, incl. RU/UZ style names), else 0."""
    try:
        name = (p.style.name or "") if p.style is not None else ""
    except Exception:
        name = ""
    low = name.strip().lower()
    if low.startswith(_HEADING_PREFIXES):
        m = re.search(r"(\d+)", low)
        return int(m.group(1)) if m else 1
    pPr = p._p.pPr
    if pPr is not None:
        ol = pPr.find(qn("w:outlineLvl"))
        if ol is not None:
            try:
                lvl = int(ol.get(qn("w:val")))
                if lvl < 9:
                    return lvl + 1
            except (TypeError, ValueError):
                pass
    return 0

def _row_texts(row):
    """Cell texts of a row; horizontally merged cells are listed once."""
    out, seen = [], []
    for c in row.cells:
        if any(c._tc is t for t in seen):
            continue
        seen.append(c._tc)
        out.append(" ".join(c.text.split()))
    return out

def table_to_lines(table):
    """Readable rows: 'Header A: value; Header B: value' when the first row looks like a header."""
    rows = []
    for row in table.rows:
        try:
            cells = _row_texts(row)
        except Exception:
            continue
        if any(cells):
            rows.append(cells)
    if not rows:
        return []
    header = rows[0]
    if len(rows) >= 2 and len(header) >= 2 and all(header):
        lines = []
        for r in rows[1:]:
            pairs = [f"{header[i] if i < len(header) else f'Column {i+1}'}: {v}"
                     for i, v in enumerate(r) if v]
            if pairs:
                lines.append("; ".join(pairs))
        return lines
    return [" | ".join(c for c in r if c) for r in rows]

def load_docx(path):
    """Return (blocks, error). Each block = {'text', 'heading', 'kind'} in document order."""
    try:
        doc = Document(path)
    except (PackageNotFoundError, zipfile.BadZipFile, KeyError, ValueError) as e:
        return None, f"not a valid .docx file ({type(e).__name__})"
    except Exception as e:
        return None, f"could not be opened ({type(e).__name__}: {str(e)[:120]})"

    blocks, stack = [], {}               # stack: heading level → heading text
    path_of = lambda: " > ".join(stack[k] for k in sorted(stack))
    for child in doc.element.body.iterchildren():
        try:
            if child.tag == qn("w:p"):
                p = Paragraph(child, doc)
                text = " ".join(p.text.split())
                if not text:
                    continue
                lvl = _heading_level(p)
                if lvl and len(text) <= 250:
                    for k in [k for k in stack if k >= lvl]:
                        del stack[k]
                    stack[lvl] = text
                    continue                 # headings become chunk context, not body text
                blocks.append({"text": text, "heading": path_of(), "kind": "para"})
            elif child.tag == qn("w:tbl"):
                for line in table_to_lines(Table(child, doc)):
                    blocks.append({"text": line, "heading": path_of(), "kind": "table"})
        except Exception:
            continue                         # skip a damaged element, keep the rest of the file
    return blocks, None

print("✅ .docx loader ready")

### Cell 6 — Chunker: ~500 tokens, ~80-token overlap, paragraph/sentence boundaries, heading metadata

In [ ]:
_SENT_RE = re.compile(r'(?:(?<=[.!?…])|(?<=[.!?…]["\'»”)\]]))\s+')

def count_tokens(text):
    return len(TOKENIZER.encode(text, add_special_tokens=False))

def split_sentences(text):
    return [s.strip() for s in _SENT_RE.split(text) if s and s.strip()]

def _split_by_words(text, max_tokens):
    """Last resort for one 'sentence' longer than a chunk: split on word boundaries."""
    pieces, cur, cur_n = [], [], 0
    for w in text.split():
        n = count_tokens(" " + w)
        if cur and cur_n + n > max_tokens:
            pieces.append(" ".join(cur)); cur, cur_n = [], 0
        cur.append(w); cur_n += n
    if cur:
        pieces.append(" ".join(cur))
    return pieces

def _tail_for_overlap(segs, overlap):
    """Trailing sentences of the finished chunk, up to `overlap` tokens, to start the next one."""
    carry, total = [], 0
    for text, n in reversed(segs):
        if total + n <= overlap:
            carry.insert(0, (text, n)); total += n
            continue
        for s in reversed(split_sentences(text)):   # take whole sentences from the end only
            ns = count_tokens(s)
            if total + ns > overlap:
                break
            carry.insert(0, (s, ns)); total += ns
        break
    return carry, total

def chunk_blocks(blocks, chunk_tokens=None, overlap=None):
    chunk_tokens = chunk_tokens or CONFIG["CHUNK_TOKENS"]
    overlap = CONFIG["CHUNK_OVERLAP"] if overlap is None else overlap

    # 1) segments: whole paragraphs when they fit, otherwise sentences
    segs = []
    for b in blocks:
        text, head = b["text"], b["heading"]
        n = count_tokens(text)
        if n <= chunk_tokens:
            segs.append((text, n, head)); continue
        for s in split_sentences(text):
            ns = count_tokens(s)
            if ns <= chunk_tokens:
                segs.append((s, ns, head))
            else:
                for piece in _split_by_words(s, chunk_tokens):
                    segs.append((piece, count_tokens(piece), head))

    # 2) pack segments into chunks; prefer a new chunk when the section heading changes
    chunks, cur, cur_n, cur_head = [], [], 0, None
    def emit():
        chunks.append({"heading": cur_head or "", "text": "\n".join(t for t, _ in cur), "n_tokens": cur_n})
    for text, n, head in segs:
        if cur:
            heading_changed = head != cur_head
            if cur_n + n > chunk_tokens or (heading_changed and cur_n >= 0.3 * chunk_tokens):
                emit()
                carry, carry_n = ([], 0) if heading_changed else _tail_for_overlap(cur, overlap)
                if carry_n + n > chunk_tokens:
                    carry, carry_n = [], 0
                cur, cur_n, cur_head = carry, carry_n, head
        if not cur:
            cur_head = head
        cur.append((text, n)); cur_n += n
    if cur:
        emit()
    return chunks

def embed_text(chunk):
    """Text that gets embedded/reranked: heading path + chunk body."""
    return f"{chunk['heading']}\n{chunk['text']}" if chunk.get("heading") else chunk["text"]

print("✅ Chunker ready")

### Cell 7 — Load the LLM in 4-bit NF4 (Qwen2.5-7B-Instruct, auto-fallback to 3B) and force greedy decoding

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,    # T4 has no fast bfloat16
)

def load_llm(names):
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    if len(names) > 1 and free_gb < CONFIG["MIN_FREE_VRAM_FOR_MAIN_LLM_GB"]:
        print(f"⚠️ Only {free_gb:.1f} GB VRAM free → using fallback {names[1]}")
        names = names[1:]
    for name in names:
        model = None
        try:
            print(f"⏳ Loading {name} (4-bit NF4) …")
            tok = AutoTokenizer.from_pretrained(name)
            model = AutoModelForCausalLM.from_pretrained(
                name,
                quantization_config=BNB_CONFIG,
                device_map={"": 0},
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
            )
            model.eval()
            return tok, model, name
        except Exception as e:
            print(f"⚠️ Could not load {name}: {type(e).__name__}: {str(e)[:300]}")
            del model
            free_memory()
    raise RuntimeError("No LLM could be loaded. Restart the runtime (Runtime → Restart session) and try again.")

def configure_greedy(model, tok):
    """temperature = 0 ⇒ greedy. Overwrite Qwen's default sampling config (do_sample=True, T=0.7)."""
    g = model.generation_config
    g.do_sample = False
    g.temperature = None
    g.top_p = None
    g.top_k = None
    g.repetition_penalty = CONFIG["REPETITION_PENALTY"]
    g.max_new_tokens = CONFIG["MAX_NEW_TOKENS"]
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    g.pad_token_id = tok.pad_token_id

_llm_names = [CONFIG["LLM_MODEL"]] + ([CONFIG["LLM_FALLBACK"]] if CONFIG["LLM_FALLBACK"] not in (None, "", CONFIG["LLM_MODEL"]) else [])
LLM_TOK, LLM, LLM_NAME = load_llm(_llm_names)
configure_greedy(LLM, LLM_TOK)
TOKENIZER = LLM_TOK                          # chunk sizes are measured in LLM tokens

CTX_LIMIT = min(getattr(LLM.config, "max_position_embeddings", 32768), CONFIG["CONTEXT_WINDOW_CAP"])
print(f"✅ LLM: {LLM_NAME} · context budget {CTX_LIMIT} tokens (prompt + {CONFIG['MAX_NEW_TOKENS']} answer)")
vram_report("after LLM")

### Cell 8 — Load the multilingual embedding model (bge-m3 on GPU, fp16, normalized vectors)

In [ ]:
from sentence_transformers import SentenceTransformer

def load_embedder():
    for name in [CONFIG["EMBED_MODEL"], CONFIG["EMBED_FALLBACK"]]:
        model = None
        try:
            print(f"⏳ Loading embedder {name} …")
            model = SentenceTransformer(name, device=DEVICE)
            model.half()
            model.max_seq_length = 1024 if "bge-m3" in name.lower() else 512
            return model, name
        except Exception as e:
            print(f"⚠️ Could not load {name}: {type(e).__name__}: {str(e)[:300]}")
            del model
            free_memory()
    raise RuntimeError("No embedding model could be loaded.")

EMBEDDER, EMBED_NAME = load_embedder()
EMBED_DIM = EMBEDDER.get_sentence_embedding_dimension()
_IS_E5 = "e5" in EMBED_NAME.lower()          # e5 needs 'query: ' / 'passage: ' prefixes

def embed_passages(texts, batch_size=16):
    if _IS_E5:
        texts = ["passage: " + t for t in texts]
    v = EMBEDDER.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=len(texts) > 32)
    return np.ascontiguousarray(v, dtype="float32")

def embed_query(q):
    q = ("query: " + q) if _IS_E5 else q
    v = EMBEDDER.encode([q], normalize_embeddings=True, convert_to_numpy=True)
    return np.ascontiguousarray(v, dtype="float32")

print(f"✅ Embedder: {EMBED_NAME} · dim {EMBED_DIM}")
vram_report("after embedder")

### Cell 9 — Optional cross-encoder reranker (bge-reranker-v2-m3), loaded on first use

In [ ]:
from sentence_transformers import CrossEncoder

RERANKER = None

def get_reranker():
    global RERANKER
    if RERANKER is None:
        print(f"⏳ Loading reranker {CONFIG['RERANK_MODEL']} …")
        RERANKER = CrossEncoder(CONFIG["RERANK_MODEL"], device=DEVICE, max_length=1024)
        RERANKER.model.half()
        vram_report("after reranker")
    return RERANKER

def unload_reranker():
    global RERANKER
    RERANKER = None
    free_memory()

if CONFIG["USE_RERANKER"]:
    get_reranker()
else:
    print("ℹ️ Reranker off by default — tick the checkbox in the UI (or set USE_RERANKER=True) to enable.")

### Cell 10 — FAISS vector store (IndexFlatIP on normalized vectors = cosine) with save/load to Drive

In [ ]:
import faiss

class VectorStore:
    def __init__(self, dim, embed_model):
        self.dim, self.embed_model = dim, embed_model
        self.index = faiss.IndexFlatIP(dim)
        self.chunks = []      # chunk dicts, aligned with FAISS ids
        self.files = {}       # sha256 → {"name", "n_chunks"}

    @property
    def size(self):
        return self.index.ntotal

    def file_names(self):
        return sorted(v["name"] for v in self.files.values())

    def add(self, chunks, vectors):
        assert len(chunks) == len(vectors)
        self.index.add(vectors)
        self.chunks.extend(chunks)

    def search(self, qvec, k):
        k = min(k, self.size)
        if k <= 0:
            return []
        scores, ids = self.index.search(qvec.reshape(1, -1), k)
        return [(self.chunks[i], float(s)) for s, i in zip(scores[0], ids[0]) if i != -1]

    def save(self, folder):
        os.makedirs(folder, exist_ok=True)
        faiss.write_index(self.index, os.path.join(folder, "index.faiss"))
        with open(os.path.join(folder, "chunks.json"), "w", encoding="utf-8") as f:
            json.dump(self.chunks, f, ensure_ascii=False)
        meta = {"embed_model": self.embed_model, "dim": self.dim, "files": self.files,
                "chunk_tokens": CONFIG["CHUNK_TOKENS"], "chunk_overlap": CONFIG["CHUNK_OVERLAP"],
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}
        with open(os.path.join(folder, "meta.json"), "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=1)

    @staticmethod
    def exists(folder):
        return all(os.path.exists(os.path.join(folder, n)) for n in ("index.faiss", "chunks.json", "meta.json"))

    @classmethod
    def load(cls, folder, expected_model):
        with open(os.path.join(folder, "meta.json"), encoding="utf-8") as f:
            meta = json.load(f)
        if meta["embed_model"] != expected_model:
            raise ValueError(f"saved index was built with {meta['embed_model']}, current embedder is "
                             f"{expected_model} — re-index instead")
        vs = cls(meta["dim"], meta["embed_model"])
        vs.index = faiss.read_index(os.path.join(folder, "index.faiss"))
        with open(os.path.join(folder, "chunks.json"), encoding="utf-8") as f:
            vs.chunks = json.load(f)
        vs.files = meta.get("files", {})
        if vs.index.ntotal != len(vs.chunks):
            raise ValueError("index and chunk metadata are out of sync")
        if (meta.get("chunk_tokens"), meta.get("chunk_overlap")) != (CONFIG["CHUNK_TOKENS"], CONFIG["CHUNK_OVERLAP"]):
            print(f"ℹ️ Saved index uses chunk {meta.get('chunk_tokens')}/{meta.get('chunk_overlap')}; "
                  f"CONFIG says {CONFIG['CHUNK_TOKENS']}/{CONFIG['CHUNK_OVERLAP']}. Reset + rebuild to apply.")
        return vs

STORE = VectorStore(EMBED_DIM, EMBED_NAME)

if DRIVE_OK and CONFIG["AUTO_LOAD_SAVED_INDEX"] and VectorStore.exists(CONFIG["DRIVE_INDEX_DIR"]):
    try:
        STORE = VectorStore.load(CONFIG["DRIVE_INDEX_DIR"], EMBED_NAME)
        print(f"✅ Loaded saved index from Drive: {STORE.size} chunks · files: {', '.join(STORE.file_names())}")
    except Exception as e:
        print(f"⚠️ Saved index not loaded: {e}")
else:
    print("ℹ️ Starting with an empty index.")

### Cell 11 — Ingestion: validate files, skip duplicates/corrupt/empty ones, chunk, embed, add, save

In [ ]:
def _unique_display_name(name, file_hash):
    """Same file name but different content → add a short hash so sources stay distinguishable."""
    if name in STORE.file_names():
        stem, ext = os.path.splitext(name)
        return f"{stem} ({file_hash[:6]}){ext}"
    return name

def ingest_files(paths, save=True):
    report, new_chunks, new_files = [], [], {}
    for path in paths:
        name = os.path.basename(path)
        if not name.lower().endswith(".docx"):
            report.append(f"⏭️ {name}: skipped — only .docx is supported (save .doc/.pdf as .docx first)")
            continue
        try:
            with open(path, "rb") as f:
                data = f.read()
        except Exception as e:
            report.append(f"❌ {name}: cannot read file ({e})"); continue
        if not data:
            report.append(f"❌ {name}: empty file (0 bytes)"); continue
        h = hashlib.sha256(data).hexdigest()
        if h in STORE.files:
            report.append(f"⏭️ {name}: duplicate of already indexed '{STORE.files[h]['name']}'"); continue
        if h in new_files:
            report.append(f"⏭️ {name}: duplicate of '{new_files[h]['name']}' in this upload"); continue

        blocks, err = load_docx(path)
        if err:
            report.append(f"❌ {name}: {err}"); continue
        if not blocks:
            report.append(f"⚠️ {name}: no extractable text (empty or image-only document)"); continue

        display = _unique_display_name(name, h)
        chunks = chunk_blocks(blocks)
        for c in chunks:
            c["file"], c["file_hash"] = display, h
        new_chunks.extend(chunks)
        new_files[h] = {"name": display, "n_chunks": len(chunks)}
        n_tables = sum(b["kind"] == "table" for b in blocks)
        report.append(f"✅ {display}: {len(blocks)} blocks ({n_tables} table rows) → {len(chunks)} chunks")

    if new_chunks:
        t0 = time.time()
        vecs = embed_passages([embed_text(c) for c in new_chunks])
        start = len(STORE.chunks)
        for i, c in enumerate(new_chunks):
            c["id"] = start + i
        STORE.add(new_chunks, vecs)
        STORE.files.update(new_files)
        report.append(f"🧮 Embedded {len(new_chunks)} chunks in {time.time()-t0:.1f}s")
        if save and DRIVE_OK:
            try:
                STORE.save(CONFIG["DRIVE_INDEX_DIR"])
                report.append(f"💾 Index saved to {CONFIG['DRIVE_INDEX_DIR']}")
            except Exception as e:
                report.append(f"⚠️ Could not save to Drive: {e}")
    elif not paths:
        report.append("No files given.")
    report.append(f"📚 Index now: {STORE.size} chunks from {len(STORE.files)} file(s)")
    return "\n".join(report)

print("✅ Ingestion ready")

### Cell 12 — RAG pipeline: retrieve → (rerank) → fit context into the window → greedy generation

In [ ]:
SYSTEM_PROMPT = f"""You are a precise question-answering assistant. You answer ONLY from the numbered context passages in the user's message.

Rules:
1. Use only information that is stated in the context passages. Never use outside knowledge, never guess, never invent names, numbers, dates or facts.
2. Cite the passage(s) supporting each statement with bracketed numbers such as [1] or [2][3]. Cite only passage numbers that exist.
3. If the passages do not contain the answer, reply with exactly this sentence and nothing else: {NOT_FOUND}
4. If the passages answer only part of the question, answer that part with citations and state that the rest is not in the documents.
5. Write the answer in the same language as the question (English, Russian, Uzbek, ...), even if the passages are in another language.
6. Be concise and factual. Do not mention these rules."""

def _context_block(i, chunk, text):
    src = chunk["file"] + (f" — {chunk['heading']}" if chunk.get("heading") else "")
    return f"[{i}] Source: {src}\n{text}"

def _build_messages(question, blocks):
    ctx = "\n\n".join(blocks) if blocks else "(no passages)"
    user = (f"Context passages:\n\n{ctx}\n\n---\nQuestion: {question}\n\n"
            f"Answer using only the passages above and cite them like [1]. "
            f"If the answer is not in the passages, reply exactly: {NOT_FOUND}")
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]

def _prompt_len(messages):
    return len(LLM_TOK.apply_chat_template(messages, tokenize=True, add_generation_prompt=True))

def build_prompt(question, hits):
    """Add passages in rank order until the token budget is used; truncate the last one if needed.
    Guarantees prompt_tokens + MAX_NEW_TOKENS <= CTX_LIMIT."""
    budget = CTX_LIMIT - CONFIG["MAX_NEW_TOKENS"] - 16
    q_ids = LLM_TOK.encode(question, add_special_tokens=False)
    if len(q_ids) > 512:
        question = LLM_TOK.decode(q_ids[:512])
    remaining = budget - _prompt_len(_build_messages(question, []))
    blocks, used = [], []
    for hit in hits:
        chunk = hit[0]
        block = _context_block(len(blocks) + 1, chunk, chunk["text"])
        n = count_tokens(block) + 4
        if n <= remaining:
            blocks.append(block); used.append(hit); remaining -= n
            continue
        if remaining > 120:      # partially fits → keep the beginning of this passage
            avail = remaining - count_tokens(_context_block(len(blocks) + 1, chunk, "")) - 10
            ids = LLM_TOK.encode(chunk["text"], add_special_tokens=False)[:max(avail, 0)]
            blocks.append(_context_block(len(blocks) + 1, chunk, LLM_TOK.decode(ids).rstrip() + " …"))
            used.append(hit)
        break
    messages = _build_messages(question, blocks)
    while blocks and _prompt_len(messages) > budget:      # exact final check
        blocks.pop(); used.pop()
        messages = _build_messages(question, blocks)
    return messages, used

def retrieve(question, top_k, use_reranker):
    if STORE.size == 0:
        return []
    n_cand = max(top_k, CONFIG["RERANK_CANDIDATES"]) if use_reranker else top_k
    hits = [(c, s, None) for c, s in STORE.search(embed_query(question), n_cand)]
    if use_reranker and hits:
        rr = get_reranker().predict([(question, embed_text(c)) for c, _, _ in hits],
                                    batch_size=8, show_progress_bar=False)
        hits = sorted(((c, s, float(r)) for (c, s, _), r in zip(hits, rr)), key=lambda h: h[2], reverse=True)
    return hits[:top_k]

@torch.inference_mode()
def generate(messages):
    prompt = LLM_TOK.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = LLM_TOK(prompt, return_tensors="pt", add_special_tokens=False).to(LLM.device)
    out = LLM.generate(
        **inputs,
        max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
        do_sample=False,                       # temperature = 0 → greedy, deterministic
        repetition_penalty=CONFIG["REPETITION_PENALTY"],
        pad_token_id=LLM_TOK.pad_token_id,
    )
    new = out[0, inputs["input_ids"].shape[1]:]
    return LLM_TOK.decode(new, skip_special_tokens=True).strip(), inputs["input_ids"].shape[1], len(new)

_NF_RE = re.compile(r"(couldn['’]?t|could not|cannot|can['’]?t) find this in the documents", re.I)

def _sources(hits, cited=()):
    return [{"rank": i + 1, "file": c["file"], "heading": c.get("heading", ""), "text": c["text"],
             "score": s, "rerank": r, "cited": (i + 1) in cited, "id": c.get("id")}
            for i, (c, s, r) in enumerate(hits)]

def answer(question, top_k=None, use_reranker=None):
    t0 = time.time()
    question = (question or "").strip()
    top_k = int(top_k or CONFIG["TOP_K"])
    use_reranker = CONFIG["USE_RERANKER"] if use_reranker is None else bool(use_reranker)
    if not question:
        return {"answer": "Please type a question.", "sources": [], "meta": ""}
    if STORE.size == 0:
        return {"answer": "The index is empty — upload .docx files and click **Build index** first.",
                "sources": [], "meta": ""}

    hits = retrieve(question, top_k, use_reranker)
    if not hits or max(h[1] for h in hits) < CONFIG["MIN_SCORE"]:
        return {"answer": NOT_FOUND, "sources": _sources(hits),
                "meta": f"best cosine below MIN_SCORE={CONFIG['MIN_SCORE']} → LLM skipped · {time.time()-t0:.1f}s"}

    messages, used = build_prompt(question, hits)
    try:
        text, n_in, n_out = generate(messages)
    except torch.cuda.OutOfMemoryError:
        free_memory()                          # retry once with half the passages
        messages, used = build_prompt(question, used[:max(1, len(used) // 2)])
        text, n_in, n_out = generate(messages)
    free_memory()

    if _NF_RE.search(text) and len(text) < 160:
        text = NOT_FOUND
    cited = {int(x) for x in re.findall(r"\[(\d+)\]", text)}
    meta = (f"{LLM_NAME.split('/')[-1]} · greedy · {n_in} prompt + {n_out} answer tokens · "
            f"{len(used)} passages · reranker {'on' if use_reranker else 'off'} · {time.time()-t0:.1f}s")
    return {"answer": text, "sources": _sources(used, cited), "meta": meta}

print("✅ RAG pipeline ready")

### Cell 13 — Upload your `.docx` files (widget) and/or pick them up from `MyDrive/rag_docs/`

In [ ]:
os.makedirs(CONFIG["LOCAL_DOCS_DIR"], exist_ok=True)
DOC_PATHS = []

try:
    from google.colab import files
    print("📤 Choose one or more .docx files (or click 'Cancel upload' to skip) …")
    _uploaded = files.upload() or {}
    for _name, _data in _uploaded.items():
        _p = os.path.join(CONFIG["LOCAL_DOCS_DIR"], os.path.basename(_name))
        with open(_p, "wb") as f:
            f.write(_data)
        DOC_PATHS.append(_p)
except Exception as e:
    print(f"ℹ️ Upload skipped ({type(e).__name__}).")

if DRIVE_OK and os.path.isdir(CONFIG["DRIVE_DOCS_DIR"]):
    _drive_docs = sorted(glob.glob(os.path.join(CONFIG["DRIVE_DOCS_DIR"], "*.docx")))
    if _drive_docs:
        print(f"📁 Found {len(_drive_docs)} .docx in {CONFIG['DRIVE_DOCS_DIR']}")
        DOC_PATHS += _drive_docs

print(f"Files to index: {[os.path.basename(p) for p in DOC_PATHS] or 'none'}")

### Cell 14 — Build the index (creates a small EN/RU/UZ demo document only if there is nothing to index)

In [ ]:
def make_demo_docx(path):
    d = Document()
    d.add_heading("Acme Ltd — Employee Handbook (demo)", 0)
    d.add_heading("Remote work", 1)
    d.add_paragraph("Employees may work remotely up to two days per week after completing their probation period. "
                    "Remote days must be agreed with the team lead by Friday of the previous week.")
    d.add_paragraph("The company reimburses home internet costs of up to 15 USD per month for remote employees.")
    d.add_heading("Equipment limits", 2)
    t = d.add_table(rows=1, cols=3)
    t.rows[0].cells[0].text, t.rows[0].cells[1].text, t.rows[0].cells[2].text = "Item", "Budget", "Replacement cycle"
    for row in [("Laptop", "1200 USD", "3 years"), ("Monitor", "250 USD", "4 years"), ("Headset", "60 USD", "2 years")]:
        cells = t.add_row().cells
        for c, v in zip(cells, row):
            c.text = v
    d.add_heading("Отпуск", 1)
    d.add_paragraph("Каждому сотруднику положено 28 календарных дней оплачиваемого ежегодного отпуска. "
                    "Заявление на отпуск подаётся не позднее чем за две недели.")
    d.add_heading("Ish vaqti", 1)
    d.add_paragraph("Ish kuni soat 09:00 da boshlanadi va 18:00 da tugaydi. Tushlik tanaffusi 13:00 dan 14:00 gacha.")
    d.save(path)
    return path

USING_DEMO = False
if not DOC_PATHS and STORE.size == 0:
    print("ℹ️ Nothing uploaded and no saved index → indexing a demo document so the notebook can run end-to-end.\n"
          "   Use 'Reset index' in the UI before uploading your real files.")
    DOC_PATHS = [make_demo_docx(os.path.join(CONFIG["LOCAL_DOCS_DIR"], "demo_handbook.docx"))]
    USING_DEMO = True

if DOC_PATHS:
    print(ingest_files(DOC_PATHS, save=not USING_DEMO))   # the demo is never saved to Drive
else:
    print(f"Using the saved index: {STORE.size} chunks.")

if STORE.chunks:
    _c = STORE.chunks[0]
    print(f"\nExample chunk → file: {_c['file']} · heading: {_c['heading'] or '—'} · {_c['n_tokens']} tokens\n"
          f"{_c['text'][:300]}…")
vram_report("after indexing")

### Cell 15 — Self-test: 3 questions in code (retrieval check + refusal check + determinism check)

In [ ]:
MY_QUESTIONS = []   # ← optional: put your own questions here to replace the auto-generated ones

def _short(t, n=160):
    t = " ".join(t.split())
    return t if len(t) <= n else t[:n] + "…"

def _auto_tests():
    if USING_DEMO:
        return [
            {"q": "How many days per week can employees work remotely?", "expect_text": "two days"},
            {"q": "Сколько дней ежегодного отпуска положено сотруднику?", "expect_text": "28"},
            {"q": "What is the recommended tyre pressure for a Boeing 747?", "expect_not_found": True},
        ]
    tests, seen = [], set()
    for i in np.linspace(0, len(STORE.chunks) - 1, num=min(len(STORE.chunks), 8), dtype=int):
        c = STORE.chunks[int(i)]
        key = (c["file"], c["heading"])
        if key in seen:
            continue
        seen.add(key)
        topic = c["heading"].split(" > ")[-1] if c["heading"] else " ".join(c["text"].split()[:12])
        tests.append({"q": f'What do the documents say about "{topic}"?', "expect_key": key, "expect_id": c["id"]})
        if len(tests) == 2:
            break
    tests.append({"q": "What is the recommended tyre pressure for a Boeing 747?", "expect_not_found": True})
    return tests

TESTS = [{"q": q} for q in MY_QUESTIONS] or _auto_tests()

for t in TESTS:
    res = answer(t["q"])
    print("=" * 100)
    print("Q:", t["q"])
    print("A:", res["answer"])
    print(f"   ({res['meta']})")
    for s in res["sources"]:
        rr = f" rerank={s['rerank']:.3f}" if s["rerank"] is not None else ""
        print(f"   [{s['rank']}] {s['file']} · {s['heading'] or '—'} · cos={s['score']:.3f}{rr}"
              f"{' · cited' if s['cited'] else ''}\n       {_short(s['text'])}")
    if "expect_text" in t:
        ok = any(t["expect_text"].lower() in s["text"].lower() for s in res["sources"])
        print(f"   Retrieval check: {'✅ expected passage retrieved' if ok else '❌ expected passage NOT retrieved'}")
    if "expect_key" in t:
        ok = any((s["file"], s["heading"]) == t["expect_key"] or s["id"] == t["expect_id"] for s in res["sources"])
        print(f"   Retrieval check: {'✅ source section retrieved' if ok else '❌ source section NOT retrieved'}")
    if t.get("expect_not_found"):
        print(f"   Refusal check:   {'✅ refused' if res['answer'] == NOT_FOUND else '⚠️ answered — inspect (or raise MIN_SCORE)'}")

# Determinism: the same question twice must give the identical answer
_q = TESTS[0]["q"]
_a1, _a2 = answer(_q)["answer"], answer(_q)["answer"]
print("=" * 100)
print(f"Determinism check: {'✅ identical answers' if _a1 == _a2 else '❌ answers differ'}")
vram_report("after self-test")

### Cell 16 — Gradio UI (`share=True`): upload → Build index → ask → answer + sources

In [ ]:
import gradio as gr

def index_status():
    if STORE.size == 0:
        return "**Index is empty.** Upload .docx files and click **Build index**."
    return (f"**{STORE.size} chunks** from **{len(STORE.files)} file(s)**: {', '.join(STORE.file_names())}  \n"
            f"Embedder `{EMBED_NAME}` · LLM `{LLM_NAME}`")

def sources_md(res):
    if not res["sources"]:
        return "_No sources._"
    out = []
    for s in res["sources"]:
        rr = f" · rerank **{s['rerank']:.3f}**" if s["rerank"] is not None else ""
        head = f" · _{s['heading']}_" if s["heading"] else ""
        cited = " · ✅ cited" if s["cited"] else ""
        body = s["text"].replace("\n", "\n> ")
        out.append(f"**[{s['rank']}] {s['file']}**{head} · cosine **{s['score']:.3f}**{rr}{cited}\n\n> {body}\n")
    return "\n---\n".join(out)

def ui_build(file_paths):
    if not file_paths:
        return "⚠️ Choose at least one .docx file first.", index_status()
    paths = [p if isinstance(p, str) else p.name for p in file_paths]
    return ingest_files(paths, save=True), index_status()

def ui_reset():
    global STORE
    STORE = VectorStore(EMBED_DIM, EMBED_NAME)
    free_memory()
    return "🗑️ In-memory index cleared (the copy on Drive is untouched until the next build/save).", index_status()

def ui_save():
    if not DRIVE_OK:
        return "⚠️ Drive is not mounted.", index_status()
    if STORE.size == 0:
        return "⚠️ Nothing to save.", index_status()
    STORE.save(CONFIG["DRIVE_INDEX_DIR"])
    return f"💾 Saved {STORE.size} chunks to {CONFIG['DRIVE_INDEX_DIR']}", index_status()

def ui_load():
    global STORE
    if not (DRIVE_OK and VectorStore.exists(CONFIG["DRIVE_INDEX_DIR"])):
        return "⚠️ No saved index found on Drive.", index_status()
    try:
        STORE = VectorStore.load(CONFIG["DRIVE_INDEX_DIR"], EMBED_NAME)
        return f"📂 Loaded {STORE.size} chunks from Drive.", index_status()
    except Exception as e:
        return f"❌ {e}", index_status()

def ui_ask(question, top_k, use_rr):
    res = answer(question, int(top_k), bool(use_rr))
    meta = f"\n\n<sub>{res['meta']}</sub>" if res["meta"] else ""
    return res["answer"] + meta, sources_md(res)

with gr.Blocks(title="Local RAG · .docx", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 📄 Local RAG over your .docx files\nEverything runs inside this Colab: "
                f"`{EMBED_NAME}` + FAISS + `{LLM_NAME}` (4-bit, greedy). Answers come only from your documents.")
    with gr.Row():
        with gr.Column(scale=1):
            files_in = gr.File(label="1 · Upload .docx files", file_count="multiple",
                               file_types=[".docx"], type="filepath")
            build_btn = gr.Button("Build index", variant="primary")
            with gr.Row():
                save_btn = gr.Button("💾 Save to Drive", size="sm")
                load_btn = gr.Button("📂 Load from Drive", size="sm")
                reset_btn = gr.Button("🗑️ Reset index", size="sm")
            log_box = gr.Textbox(label="Log", lines=8, interactive=False)
            status_md = gr.Markdown(index_status())
        with gr.Column(scale=2):
            q_box = gr.Textbox(label="2 · Your question (English / Русский / O‘zbek)", lines=2,
                               placeholder="e.g. How many vacation days do employees get?")
            with gr.Row():
                topk_sl = gr.Slider(1, 20, value=CONFIG["TOP_K"], step=1, label="top-k")
                rr_cb = gr.Checkbox(value=CONFIG["USE_RERANKER"], label="Use reranker (bge-reranker-v2-m3)")
            ask_btn = gr.Button("Ask", variant="primary")
            gr.Markdown("### Answer")
            answer_md = gr.Markdown()
            gr.Markdown("### Sources used")
            sources_box = gr.Markdown()

    build_btn.click(ui_build, [files_in], [log_box, status_md])
    reset_btn.click(ui_reset, None, [log_box, status_md])
    save_btn.click(ui_save, None, [log_box, status_md])
    load_btn.click(ui_load, None, [log_box, status_md])
    ask_btn.click(ui_ask, [q_box, topk_sl, rr_cb], [answer_md, sources_box])
    q_box.submit(ui_ask, [q_box, topk_sl, rr_cb], [answer_md, sources_box])

demo.queue(default_concurrency_limit=1).launch(share=True, debug=False, show_error=True)

### Cell 17 — Optional utilities: switch LLM without restarting, stop the UI

In [ ]:
def switch_llm(name):
    """e.g. switch_llm('Qwen/Qwen2.5-3B-Instruct'). The index stays valid (same tokenizer family)."""
    global LLM, LLM_TOK, LLM_NAME, TOKENIZER, CTX_LIMIT
    LLM = None
    free_memory()
    vram_report("after unloading LLM")
    LLM_TOK, LLM, LLM_NAME = load_llm([name])
    configure_greedy(LLM, LLM_TOK)
    TOKENIZER = LLM_TOK
    CTX_LIMIT = min(getattr(LLM.config, "max_position_embeddings", 32768), CONFIG["CONTEXT_WINDOW_CAP"])
    vram_report(f"after loading {name}")

# switch_llm("Qwen/Qwen2.5-3B-Instruct")   # ← uncomment to use
# unload_reranker()                          # ← frees ~1.1 GB if you enabled the reranker
# demo.close()                               # ← stops the Gradio server